# Résoudre `7_big` avec OR-Tools
Dernier jeu de données : **50 000 modèles et 250 000 datasets**. Le commentaire de l'instance : *« We're in the endgame now »*.

Pas de nouvelle idée de modélisation ici : `7_big` ressemble à une version agrandie de `2_medium`. Le défi est la **taille**. On reprend le pipeline des notebooks 14 et 15 (contrôle préalable, modèle agrégé, énergie en milliards, file d'attente unique, complétion, réparation) et on regarde ce qui tient à cette échelle, et ce qu'il faut alléger.

**Prérequis** : `datasets/7_big.json`, `test_solution.py`, `pandas`, `ortools`.

**Durée d'exécution** : environ 25 minutes (4 pour le contrôle préalable, 10 pour le solveur, 6 pour l'attribution et la réparation).

## 1. Charger l'instance et le repère

In [1]:
from pathlib import Path
from collections import defaultdict, Counter
import sys, json, time, bisect, random
import pandas as pd
from ortools.linear_solver import pywraplp
pd.set_option("display.width", 150)

ROOT_DIR = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(ROOT_DIR))
from test_solution import getSolutionScore

DATASET = ROOT_DIR / "datasets" / "7_big.json"
DATASET_TXT = DATASET.read_text()
instance = json.loads(DATASET_TXT)
energy_cap = instance["energyCap"]
print("commentaire de l'instance :", instance["comment"])

VALUE  = {m["id"]: m["value"] for m in instance["models"]}
ENERGY = {m["id"]: m["energyCost"] for m in instance["models"]}
NEEDS  = {m["id"]: [(t, b["lowerBound"], b["upperBound"]) for t, b in m["dataRequirements"].items()] for m in instance["models"]}
SIZE   = {d["id"]: d["size"] for d in instance["datasets"]}
TYPE   = {d["id"]: d["dataType"] for d in instance["datasets"]}
COPY   = {d["id"]: d["isCopyrighted"] for d in instance["datasets"]}
IMPOSSIBLE = {m for m, needs in NEEDS.items() if any(lo > hi for _, lo, hi in needs)}
LOWER = {m: {t: lo for t, lo, hi in needs if lo <= hi} for m, needs in NEEDS.items()}
MONO = {m: needs[0][0] for m, needs in NEEDS.items() if len(needs) == 1 and m not in IMPOSSIBLE}
FREE_SUPPLY  = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t and COPY[d] == 0) for t in "ntic"}
TOTAL_SUPPLY = {t: sum(SIZE[d] for d in SIZE if TYPE[d] == t) for t in "ntic"}

WINNER = 5_822_561
print(f"{len(VALUE)} modèles, {len(SIZE)} datasets | gagnant du concours : {WINNER:,}")

commentaire de l'instance : We're in the endgame now.


50000 modèles, 250000 datasets | gagnant du concours : 5,822,561


## 2. Ce qui change

In [2]:
print(f"énergie de tous les modèles : {sum(ENERGY.values()) / energy_cap:.2f} × le plafond")
print(f"datasets sous droits : {sum(COPY.values()) / len(COPY):.0%}")
print(f"modèles à énergie quasi nulle : {sum(e <= 10 for e in ENERGY.values())}")
print(f"modèles à un seul besoin (sources possibles) : {len(MONO)} | modèles impossibles : {len(IMPOSSIBLE)}")
share = energy_cap / sum(ENERGY.values())                     # part des modèles que l'énergie permet
demand = {t: sum(LOWER[m].get(t, 0) for m in VALUE) for t in "ntic"}
pd.DataFrame({"offre_totale": TOTAL_SUPPLY, "offre_libre": FREE_SUPPLY,
              "demande_probable": {t: round(demand[t] * share) for t in "ntic"},
              "demande_probable_sur_offre_libre": {t: round(demand[t] * share / FREE_SUPPLY[t], 2) for t in "ntic"}})

énergie de tous les modèles : 5.82 × le plafond
datasets sous droits : 66%
modèles à énergie quasi nulle : 3055
modèles à un seul besoin (sources possibles) : 21094 | modèles impossibles : 1043


,offre_totale,offre_libre,demande_probable,demande_probable_sur_offre_libre
n,311335250,105406037,101228920,0.96
t,311180236,105427627,102015459,0.97
i,312781383,105818003,100837797,0.95
c,313095242,106153061,100267925,0.94


**Lecture**
- L'énergie de tous les modèles fait 5,8 fois le plafond : on entraînera environ un modèle sur six. **L'énergie limite.**
- Les modèles que l'énergie permet demanderaient à peu près autant de données que l'offre **libre**. Les données libres seront donc juste suffisantes, comme sur `2_medium` : c'est elles qui décideront combien de modèles sont à pleine valeur.
- Les sources coûtent de l'énergie (la leur) sans rien rapporter, et les données totales ne manquent pas. Elles ne devraient servir qu'aux modèles impossibles. On le vérifie en partie 4.

## 3. Contrôle préalable
Comme dans le notebook 15 : un modèle dont un besoin ne peut pas être rempli en données libres seules est interdit à pleine valeur. Avec 250 000 datasets, ce contrôle prend environ 4 minutes. Ici, aucun modèle n'est écarté.

In [3]:
t0 = time.time()
class Pool:
    def __init__(self, ids):
        self.ids = sorted(ids, key=SIZE.get)
        self.sizes = [SIZE[d] for d in self.ids]
    def take(self, i):
        self.sizes.pop(i)
        return self.ids.pop(i)
    def put(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        self.sizes.insert(i, SIZE[d]); self.ids.insert(i, d)
    def remove(self, d):
        i = bisect.bisect_left(self.sizes, SIZE[d])
        while self.ids[i] != d:
            i += 1
        self.take(i)

def find_pair(sizes, a, b):
    i, j, best = 0, bisect.bisect_right(sizes, b) - 1, None
    while i < j:
        s = sizes[i] + sizes[j]
        if s > b:
            j -= 1
        elif s < a:
            i += 1
        else:
            if best is None or s < best[0]:
                best = (s, i, j)
            j -= 1
    return best

def fill(pool, lower, upper):
    taken, total = [], 0
    while total < lower:
        a, b = lower - total, upper - total
        i = bisect.bisect_left(pool.sizes, a)
        if i < len(pool.sizes) and pool.sizes[i] <= b:
            total += pool.sizes[i]; taken.append(pool.take(i)); continue
        pair = find_pair(pool.sizes, a, b) if pool.sizes and a <= 2 * pool.sizes[-1] else None
        if pair:
            _, i, j = pair
            total += pool.sizes[i] + pool.sizes[j]
            taken.append(pool.take(j)); taken.append(pool.take(i)); continue
        j = bisect.bisect_right(pool.sizes, b) - 1
        if j < 0:
            break
        total += pool.sizes[j]; taken.append(pool.take(j))
    if lower <= total <= upper:
        return taken
    for d in taken:
        pool.put(d)
    return None


pools = {(t, c): Pool([d for d in SIZE if TYPE[d] == t and COPY[d] == c]) for t in "ntic" for c in (0, 1)}
NO_FULL = set()
for m, needs in NEEDS.items():
    for t, lo, hi in needs:
        if lo <= hi:
            got = fill(pools[(t, 0)], lo, hi)
            if got is None:
                NO_FULL.add(m)
            else:
                for d in got:
                    pools[(t, 0)].put(d)
print("modèles interdits à pleine valeur (un besoin impossible en données libres seules) :", sorted(NO_FULL))
print(f"{time.time() - t0:.0f} s")

modèles interdits à pleine valeur (un besoin impossible en données libres seules) : []
245 s


## 4. Le modèle
Le même que dans le notebook 15. À cette échelle, le choix de `possible_sources` compte :
- avec `possible_sources=False`, les variables $c_{i,t}$ n'existent que pour les besoins impossibles : environ 120 000 variables ;
- avec `possible_sources=True`, on ajoute une variable par besoin remplissable : environ 100 000 variables de plus.

Pendant la préparation, j'ai lancé les deux versions 10 minutes chacune :

| Version | Solution après 10 min | Borne |
|---|---|---|
| sources pour les besoins impossibles seulement | 5 823 205 | 5 823 268 |
| sources partout | 4 348 180 | 37 340 527 |

Le gros modèle n'a même pas dépassé les premières étapes de la recherche : sa borne est encore très lâche. On garde donc le modèle léger. C'est le contraire de `5_energy` et `6_shortage`, où les données manquaient et où les sources étaient indispensables. **Le meilleur modèle n'est pas le plus complet, c'est celui qui suffit.**

In [4]:
def solve_model(seconds, possible_sources=True, total_margin=1.0):
    t0 = time.time()
    S = pywraplp.Solver.CreateSolver("SCIP")
    x = {m: S.BoolVar(f"x{m}") for m in VALUE if m not in IMPOSSIBLE and m not in NO_FULL}   # pleine valeur
    y = {m: S.BoolVar(f"y{m}") for m in VALUE}                              # demi-valeur
    w = {s: S.BoolVar(f"w{s}") for s in MONO}                               # source
    c = {}                                                                  # besoin couvert par une source
    for m, needs in NEEDS.items():
        for t, lo, hi in needs:
            if lo > hi:
                c[(m, t)] = y[m]
            elif possible_sources:
                c[(m, t)] = S.BoolVar(f"c{m}_{t}")
                S.Add(c[(m, t)] <= y[m])
    for m in VALUE:
        S.Add((x[m] if m in x else 0) + y[m] + (w[m] if m in w else 0) <= 1)   # un seul rôle
    E = 1e9                                                                 # énergie en milliards
    S.Add(sum(ENERGY[m] / E * x[m] for m in x) + sum(ENERGY[m] / E * y[m] for m in y)
          + sum(ENERGY[s] / E * w[s] for s in w) <= energy_cap / E)
    for t in "ntic":
        S.Add(sum(var for (m, tt), var in c.items() if tt == t) == sum(w[s] for s in w if MONO[s] == t))
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m]) <= FREE_SUPPLY[t])
        S.Add(sum(LOWER[m][t] * x[m] for m in x if t in LOWER[m])
              + sum(LOWER[m][t] * (y[m] - c.get((m, t), 0)) for m in y if t in LOWER[m])
              + sum(LOWER[s][t] * w[s] for s in w if MONO[s] == t) <= TOTAL_SUPPLY[t] * total_margin)
    S.Maximize(sum(VALUE[m] * x[m] for m in x) + sum((VALUE[m] // 2) * y[m] for m in y))
    S.SetTimeLimit(seconds * 1000)
    params = pywraplp.MPSolverParameters()
    params.SetDoubleParam(params.RELATIVE_MIP_GAP, 0.0)
    status = S.Solve(params)
    val = lambda v: round(v.solution_value())
    sol = dict(full=[m for m in x if val(x[m])], half=[m for m in y if val(y[m])],
               sources=[s for s in w if val(w[s])], covered=[(m, t) for (m, t), var in c.items() if val(var)])
    obj, bound = S.Objective().Value(), S.Objective().BestBound()
    info = dict(statut="OPTIMAL" if status == pywraplp.Solver.OPTIMAL else "FEASIBLE", objectif=round(obj),
                borne=round(bound, 1), écart_prouvé=round(bound - obj, 1), secondes=round(time.time() - t0),
                pleins=len(sol["full"]), demi=len(sol["half"]), sources=len(sol["sources"]))
    used = sum(ENERGY[m] for m in sol["full"] + sol["half"] + sol["sources"])
    assert used <= energy_cap, "énergie exacte dépassée"
    return sol, info

solver_runs = {}

## 5. Résoudre

In [5]:
solver_runs = {}
sol, solver_runs["sources : besoins impossibles seulement"] = solve_model(600, possible_sources=False)
pd.DataFrame(solver_runs).T

,statut,objectif,borne,écart_prouvé,secondes,pleins,demi,sources
sources : besoins impossibles seulement,FEASIBLE,5823205,5823267.7,62.7,605,11719,4551,188


**Lecture** : en 10 minutes, la sélection du solveur dépasse le gagnant sur le papier, avec une borne à quelques dizaines de points au-dessus. Sur un problème de 50 000 modèles, SCIP reste très efficace, parce que la relaxation linéaire est presque entière (comme sur `2_medium`).

## 6. Attribuer, compléter, réparer
Les fonctions viennent du notebook 15. Deux adaptations à la taille :
- une attribution prend maintenant **plus d'une minute** (250 000 datasets). On n'essaie donc que les deux ordres de référence, pas 200 ordres bruités. Pendant la préparation, trois ordres bruités ont d'ailleurs fait **moins bien** que l'ordre « petits besoins » ;
- la réparation revient à une limite de **20 partenaires** : avec environ 4 500 modèles à demi-valeur et 11 700 pleins, tester tous les partenaires prendrait des heures.

In [6]:
def give(pools, model, needs, statuses):
    got = []
    for t, lo, hi in needs:
        res = None
        for s in statuses:
            res = fill(pools[(t, s)], lo, hi)
            if res is not None:
                break
        if res is None:
            for d in got:
                pools[(TYPE[d], COPY[d])].put(d)
            return None
        got += res
    return [[int(d), int(model)] for d in got]

def pack(sol, order_key):
    """order_key(modèle, besoins à remplir) -> clé de tri. Rend (plan, nombre de modèles ratés)."""
    pools = {(t, c): Pool([d for d in SIZE if TYPE[d] == t and COPY[d] == c]) for t in "ntic" for c in (0, 1)}
    covered = defaultdict(set)
    for m, t in sol["covered"]:
        covered[m].add(t)
    jobs = [(m, NEEDS[m], (0,), "plein") for m in sol["full"]]
    jobs += [(m, [n for n in NEEDS[m] if n[0] not in covered[m]], (1, 0), "demi") for m in sol["half"]]
    jobs += [(s, NEEDS[s], (1, 0), "source") for s in sol["sources"]]
    data, failed = {}, []
    for m, needs, statuses, kind in sorted(jobs, key=lambda j: order_key(j[0], j[1])):
        got = give(pools, m, needs, statuses)
        if got is None:
            failed.append((m, kind))
        else:
            data[m] = got
    for m, kind in failed:                                   # seconde chance à demi-valeur pour les pleins
        if kind == "plein":
            got = give(pools, m, NEEDS[m], (1, 0))
            if got is not None:
                data[m] = got
    # Appariement sources -> besoins couverts, type par type
    available = defaultdict(list)
    for s in sol["sources"]:
        if s in data:
            available[MONO[s]].append(s)
    model_mappings = []
    for m in sol["half"]:
        if m not in data:
            continue
        pairs = []
        for t in covered[m]:
            if available[t]:
                pairs.append((available[t].pop(), m))
        if len(pairs) == len(covered[m]):
            model_mappings += pairs
        else:                                                # cible incomplète : on la retire
            for s, _ in pairs:
                available[MONO[s]].append(s)
            data.pop(m)
    for t, unused in available.items():                      # sources sans cible : on les retire
        for s in unused:
            data.pop(s)
    plan = {"dataMappings": [p for got in data.values() for p in got],
            "modelMappings": [[int(s), int(m)] for s, m in model_mappings]}
    return plan, len(failed)

In [7]:
class State:
    """Solution modifiable, avec journal d'annulation. Sources et cibles sont gelées."""
    def __init__(self, plan):
        frozen_models = {x for pair in plan["modelMappings"] for x in pair}
        self.frozen_data = [p for p in plan["dataMappings"] if p[1] in frozen_models]
        self.frozen_sources = plan["modelMappings"]
        self.frozen = frozen_models
        self.data = {}
        for d, m in plan["dataMappings"]:
            if m not in frozen_models:
                self.data.setdefault(m, []).append(d)
        used = {d for d, _ in plan["dataMappings"]}
        self.pools = {(t, c): Pool([d for d in SIZE if d not in used and TYPE[d] == t and COPY[d] == c])
                      for t in "ntic" for c in (0, 1)}
        self.full = {m: not any(COPY[d] for d in ds) for m, ds in self.data.items()}
        self.energy = sum(ENERGY[m] for m in self.data) + sum(ENERGY[m] for m in frozen_models)
        self.log = []

    def remove(self, m):
        ds, full = self.data.pop(m), self.full.pop(m)
        for d in ds:
            self.pools[(TYPE[d], COPY[d])].put(d)
        self.energy -= ENERGY[m]
        self.log.append(("remove", m, ds, full))

    def add(self, m, statuses):
        if self.energy + ENERGY[m] > energy_cap:      # ici, l'énergie limite
            return False
        if statuses == (0,) and m in NO_FULL:
            return False
        got = give(self.pools, m, NEEDS[m], statuses)
        if got is None:
            return False
        self.data[m] = [d for d, _ in got]
        self.full[m] = not any(COPY[d] for d in self.data[m])
        self.energy += ENERGY[m]
        self.log.append(("add", m, None, None))
        return True

    def undo(self):
        while self.log:
            op, m, ds, full = self.log.pop()
            if op == "add":
                for d in self.data.pop(m):
                    self.pools[(TYPE[d], COPY[d])].put(d)
                self.full.pop(m); self.energy -= ENERGY[m]
            else:
                for d in ds:
                    self.pools[(TYPE[d], COPY[d])].remove(d)
                self.data[m], self.full[m] = ds, full
                self.energy += ENERGY[m]

    def plan(self):
        mappings = [[int(d), int(m)] for m, ds in self.data.items() for d in ds] + self.frozen_data
        return {"dataMappings": mappings, "modelMappings": self.frozen_sources}

TYPES_OF = {m: {t for t, _, _ in needs} for m, needs in NEEDS.items()}
gain_full = lambda m: VALUE[m] - VALUE[m] // 2

def complete(s):
    candidates = [m for m in VALUE if m not in s.data and m not in s.frozen and m not in IMPOSSIBLE]
    candidates.sort(key=lambda m: -VALUE[m] / max(1, sum(LOWER[m].values())))
    added = 0
    for m in candidates:
        s.log = []
        if s.add(m, (0,)) or s.add(m, (1, 0)):
            added += 1
    return added

def repair(s, max_partners=100_000):
    n_repairs, improved = 0, True
    while improved:
        improved = False
        for b in sorted([m for m in s.data if not s.full[m]], key=gain_full, reverse=True):
            if b not in s.data or s.full[b]:
                continue
            s.log = []; s.remove(b)
            if s.add(b, (0,)):
                n_repairs += 1; improved = True; continue
            s.undo()
            partners = sorted([a for a in s.data if s.full[a] and TYPES_OF[a] & TYPES_OF[b] and gain_full(a) < gain_full(b)],
                              key=gain_full)
            for a in partners[:max_partners]:
                s.log = []; s.remove(a); s.remove(b)
                if s.add(b, (0,)) and s.add(a, (1, 0)):
                    n_repairs += 1; improved = True; break
                s.undo()
    return n_repairs



rows, best = [], (0, None, None)
orders = {"petits besoins": lambda m, needs: sum(lo for t, lo, hi in needs),
          "plus contraints": lambda m, needs: (min((hi for t, lo, hi in needs), default=0), sum(lo for t, lo, hi in needs))}
for name, key in orders.items():
    t0 = time.time()
    plan, n_failed = pack(sol, key)
    score0 = getSolutionScore(json.dumps(plan), DATASET_TXT, show_details=False)[0]
    t1 = time.time()
    state = State(plan)
    added = complete(state)
    n = repair(state, max_partners=20)
    final = state.plan()
    score = getSolutionScore(json.dumps(final), DATASET_TXT, show_details=False)[0]
    rows.append(dict(ordre=name, ratés=n_failed, après_attribution=score0, ajouts=added, réparations=n, score=score,
                     écart_au_gagnant=score - WINNER, secondes_attribution=round(t1 - t0), secondes_réparation=round(time.time() - t1)))
    if score > best[0]:
        best = (score, name, final)
pd.DataFrame(rows).set_index("ordre")

,ratés,après_attribution,ajouts,réparations,score,écart_au_gagnant,secondes_attribution,secondes_réparation
ordre,,,,,,,,
petits besoins,13,5821868,6,13,5822283,-278,85,111
plus contraints,4,5821238,6,10,5822383,-178,92,75


**Lecture**
- L'ordre « plus contraints » fait échouer moins de modèles et, après réparation, donne le meilleur résultat.
- Après attribution, complétion et réparation, on arrive à quelques centaines de points du gagnant, soit plus de **99,99 %** de son score.

Une piste essayée sans succès : une marge de 0,2 % sur les **données libres**. Plus aucun modèle n'échoue, mais la sélection du solveur perd environ **4 200 points**. Ici, les données libres sont si précieuses que chaque dixième de pourcent vaut environ 2 000 points : mieux vaut les utiliser toutes et réparer les quelques échecs.

## 7. Vérification et fichier de solution

In [8]:
best_score, best_name, best_plan = best
score, valid, message = getSolutionScore(json.dumps(best_plan), DATASET_TXT, show_details=True)
print("meilleur ordre :", best_name, "| score officiel =", score, "| valide =", valid, message)
print(f"écart au gagnant : {score - WINNER:+,} ({score / WINNER:.4%})")

out_dir = ROOT_DIR / "solutions"; out_dir.mkdir(exist_ok=True)
out = out_dir / "7_big_main.json"
current = getSolutionScore(out.read_text(), DATASET_TXT, show_details=False)[0] if out.exists() else 0
if valid and score > current:
    out.write_text(json.dumps(best_plan))
    print(f"7_big_main.json mis à jour : {current:,} -> {score:,}")
else:
    print(f"7_big_main.json inchangé ({current:,})")


Total value: 5822383
Energy used: 4328798518960237 / 4328801389315026 (100.00%)
Number of used models: 16464 / 50000 (32.93%)
  Number of devalued models: 4741 (28.80%)
  Number of models used as data source: 188 (1.14%)
Number of used datasets: 107340 / 250000 (42.94%)
meilleur ordre : plus contraints | score officiel = 5822383 | valide = True 
écart au gagnant : -178 (99.9969%)


7_big_main.json inchangé (5,822,383)


## 8. Bilan des sept jeux de données
Au concours, chaque jeu de données rapporte `1 000 000 × score / meilleur score`. Voici où nous en sommes, avec les meilleures solutions obtenues dans les notebooks 08 à 16.

In [9]:
final = {   # meilleurs scores obtenus dans les notebooks 08 à 16
    "1_example":  (12_946,    12_946,    "08 : énumération exhaustive"),
    "2_medium":   (272_224,   272_322,   "11 : SCIP + réparation + boucle"),
    "3_free":     (523_758,   523_758,   "12 : SCIP, optimum prouvé"),
    "4_precise":  (1_031_910, 1_031_941, "13 : SCIP (énergie à l'échelle) + 40 ordres"),
    "5_energy":   (3_785_420, 3_786_108, "14 : sources agrégées + piège numérique"),
    "6_shortage": (668_777,   669_798,   "15 : sources + marge sur l'offre totale"),
    "7_big":      (score,     WINNER,    "16 : pipeline allégé"),
}
recap = pd.DataFrame(final, index=["notre score", "gagnant", "méthode"]).T
recap["écart"] = recap["notre score"] - recap["gagnant"]
recap["points_classement"] = (1_000_000 * recap["notre score"] / recap["gagnant"]).round().astype(int)
display(recap)
print(f"total : {recap.points_classement.sum():,} points sur 7 000 000 possibles ({recap.points_classement.sum() / 7_000_000:.3%})")

,notre score,gagnant,méthode,écart,points_classement
1_example,12946,12946,08 : énumération exhaustive,0,1000000
2_medium,272224,272322,11 : SCIP + réparation + boucle,-98,999640
3_free,523758,523758,"12 : SCIP, optimum prouvé",0,1000000
4_precise,1031910,1031941,13 : SCIP (énergie à l'échelle) + 40 ordres,-31,999970
5_energy,3785420,3786108,14 : sources agrégées + piège numérique,-688,999818
6_shortage,668777,669798,15 : sources + marge sur l'offre totale,-1021,998476
7_big,5822383,5822561,16 : pipeline allégé,-178,999969


total : 6,997,873 points sur 7 000 000 possibles (99.970%)


## À retenir (sur l'ensemble du parcours)
1. **Regarder les données avant de choisir l'algorithme.** Chaque instance avait sa contrainte active : l'énergie, les données libres, la précision, les données totales, ou tout à la fois. Le modèle a dû suivre à chaque fois.
2. **L'exact et l'heuristique se complètent.** Le solveur choisit les modèles ; des règles gloutonnes attribuent les datasets ; une réparation corrige l'écart entre les deux.
3. **Une borne est un instrument de mesure.** Elle a dit combien il restait à gagner, et elle a révélé deux erreurs : une hypothèse fausse sur les sources (notebook 10) et un faux optimum numérique (notebook 14).
4. **Un solveur se règle et peut se tromper.** Tolérance d'arrêt (notebook 12), contraintes oubliées (notebook 13), coefficients trop grands (notebook 14), limite de temps non reproductible (notebook 13).
5. **À grande échelle, on allège.** Moins d'ordres d'attribution, une réparation limitée, le modèle le plus léger qui suffit. Le meilleur pipeline est celui qu'on peut exécuter.